# Synthetic market-data comparison

Compare the full-year price, risk, volume, correlation, and intraday patterns of all ten stocks. The selected symbol and month are also shown as a detailed candlestick drill-down. The notebook reads candle partitions directly with DuckDB; PostgreSQL and the full tick archive are not loaded.

## 1. Choose a month and drill-down stock

All ten symbols are always included in comparison charts. `SELECTED_SYMBOL` controls only the detailed chart and daily table.

In [16]:
SELECTED_MONTH = 'January'
SELECTED_SYMBOL = 'AAPL'
BAR_MINUTES = 15

## 2. Load all ten stocks for the full year

In [17]:
from __future__ import annotations

import calendar
import json
from pathlib import Path

import duckdb
import pandas as pd
import plotly.graph_objects as go
from IPython.display import HTML, display
from plotly.subplots import make_subplots

DATASET_NAME = 'synthetic-market-data-2026-v1'
search_roots = [Path.cwd(), *Path.cwd().parents]
candidates = []
for root in search_roots:
    candidates.extend([root / DATASET_NAME, root / 'apps/market-data/db/seeds' / DATASET_NAME])
DATASET = next((path.resolve() for path in candidates if (path / 'manifest.json').is_file()), None)
if DATASET is None:
    searched = '\n'.join(f'  - {path.resolve()}' for path in candidates)
    raise FileNotFoundError(f'{DATASET_NAME}/manifest.json was not found. Searched:\n{searched}')

manifest = json.loads((DATASET / 'manifest.json').read_text(encoding='utf-8'))
symbols = [item['symbol'] if isinstance(item, dict) else item for item in manifest.get('symbols', [])]
symbol = str(SELECTED_SYMBOL).strip().upper()
if symbol not in symbols:
    raise ValueError(f'Unknown symbol {symbol!r}. Choose one of: {", ".join(symbols)}')

month_lookup = {}
for number in range(1, 13):
    month_lookup.update({str(number): number, f'{number:02d}': number, calendar.month_name[number].lower(): number, calendar.month_abbr[number].lower(): number})
month_key = str(SELECTED_MONTH).strip().lower()
if month_key not in month_lookup:
    raise ValueError('Month must be 1-12, a month name, or a three-letter abbreviation.')
month = month_lookup[month_key]
year = int(manifest.get('config', {}).get('year', 2026))
year_label = str(year)
month_label = f'{calendar.month_name[month]} {year}'
month_prefix = f'{year}-{month:02d}'
month_entries = [item for item in manifest.get('candle_files', []) if item['day'].startswith(month_prefix)]
if not month_entries:
    raise ValueError(f'No candle partitions are recorded for {month_label}.')
year_entries = [item for item in manifest.get('candle_files', []) if item['day'].startswith(year_label)]
year_files = [str(DATASET / item['name']) for item in year_entries]
missing_files = [path for path in year_files if not Path(path).is_file()]
if missing_files:
    raise FileNotFoundError(f'Missing candle partition: {missing_files[0]}')
if not isinstance(BAR_MINUTES, int) or BAR_MINUTES < 1 or BAR_MINUTES > 390:
    raise ValueError('BAR_MINUTES must be an integer from 1 through 390.')

db = duckdb.connect()
query = f'''
    SELECT symbol, time_bucket(INTERVAL '{BAR_MINUTES} minutes', to_timestamp(t)) AS timestamp,
           arg_min(open, t)::DOUBLE AS open, max(high)::DOUBLE AS high,
           min(low)::DOUBLE AS low, arg_max(close, t)::DOUBLE AS close,
           sum(volume)::BIGINT AS volume, sum(trade_count)::BIGINT AS trade_count
    FROM read_parquet(?)
    GROUP BY symbol, 2 ORDER BY 2, symbol
'''
all_candles = db.execute(query, [year_files]).fetch_df()
if all_candles.empty:
    raise ValueError(f'No candle rows found for {year_label}.')
all_candles['timestamp'] = pd.to_datetime(all_candles['timestamp'], utc=True).dt.tz_convert('America/Chicago')
all_candles['day'] = all_candles['timestamp'].dt.date
all_candles['time'] = all_candles['timestamp'].dt.strftime('%H:%M')
candles = all_candles.loc[all_candles['symbol'].eq(symbol) & all_candles['timestamp'].dt.month.eq(month)].copy()
display(HTML(f'<b>{year_label} full-year comparison</b> | {len(symbols)} symbols: {", ".join(symbols)} | {len(year_entries)} daily partitions | {len(all_candles):,} {BAR_MINUTES}-minute bars | drill-down: {symbol}, {month_label}'))

## 3. Relative price paths

Each stock starts at 0%, making differently priced stocks directly comparable.

In [18]:
all_candles['normalized_return_pct'] = all_candles.groupby('symbol')['close'].transform(lambda values: (values / values.iloc[0] - 1) * 100)
normalized_figure = go.Figure()
for ticker in symbols:
    stock = all_candles.loc[all_candles['symbol'].eq(ticker)]
    normalized_figure.add_trace(go.Scatter(x=stock['timestamp'], y=stock['normalized_return_pct'], mode='lines', name=ticker, line={'width': 2}, hovertemplate='%{x}<br>%{y:+.2f}%<extra>' + ticker + '</extra>'))
normalized_figure.add_hline(y=0, line_width=1, line_dash='dot', line_color='#64748b')
normalized_figure.update_layout(title=f'All stocks: normalized performance in {year_label}', height=620, template='plotly_white', hovermode='x unified', yaxis_title='Return from first bar (%)', xaxis_title=None, legend={'orientation': 'h', 'yanchor': 'bottom', 'y': 1.02, 'x': 0})
normalized_figure.show()

## 4. Return, risk, and activity comparison

Annual return measures the full period. Volatility is the standard deviation of daily open-to-close returns, and maximum drawdown is the largest decline from a prior daily closing peak.

In [19]:
daily_all = (all_candles.groupby(['symbol', 'day'], as_index=False)
             .agg(open=('open', 'first'), high=('high', 'max'), low=('low', 'min'), close=('close', 'last'), volume=('volume', 'sum'), trade_count=('trade_count', 'sum')))
daily_all['daily_return_pct'] = (daily_all['close'] / daily_all['open'] - 1) * 100
daily_all['range_pct'] = (daily_all['high'] / daily_all['low'] - 1) * 100
daily_all['drawdown_pct'] = (daily_all['close'] / daily_all.groupby('symbol')['close'].cummax() - 1) * 100
comparison = (daily_all.groupby('symbol', as_index=False)
              .agg(start_price=('open', 'first'), end_price=('close', 'last'), avg_daily_return_pct=('daily_return_pct', 'mean'), daily_volatility_pct=('daily_return_pct', 'std'), avg_range_pct=('range_pct', 'mean'), max_drawdown_pct=('drawdown_pct', 'min'), avg_daily_volume=('volume', 'mean')))
comparison['annual_return_pct'] = (comparison['end_price'] / comparison['start_price'] - 1) * 100
positive_days = daily_all.assign(positive=daily_all['daily_return_pct'].gt(0)).groupby('symbol')['positive'].mean() * 100
comparison['positive_days_pct'] = comparison['symbol'].map(positive_days)
comparison = comparison.sort_values('annual_return_pct', ascending=False).reset_index(drop=True)
comparison = comparison[['symbol', 'annual_return_pct', 'avg_daily_return_pct', 'daily_volatility_pct', 'avg_range_pct', 'max_drawdown_pct', 'positive_days_pct', 'avg_daily_volume']]
display(comparison.style.format({'annual_return_pct': '{:+.2f}%', 'avg_daily_return_pct': '{:+.2f}%', 'daily_volatility_pct': '{:.2f}%', 'avg_range_pct': '{:.2f}%', 'max_drawdown_pct': '{:.2f}%', 'positive_days_pct': '{:.0f}%', 'avg_daily_volume': '{:,.0f}'}))

,symbol,annual_return_pct,avg_daily_return_pct,daily_volatility_pct,avg_range_pct,max_drawdown_pct,positive_days_pct,avg_daily_volume
0,META,+42.14%,+0.14%,1.25%,1.40%,-18.44%,56%,"18,153,829"
1,GOOGL,+35.02%,+0.12%,0.91%,1.15%,-27.86%,59%,"31,767,993"
2,XOM,+34.08%,+0.12%,0.87%,1.06%,-27.61%,58%,"17,018,918"
3,AAPL,+32.35%,+0.11%,0.78%,0.97%,-27.08%,58%,"62,401,606"
4,NVDA,+28.89%,+0.11%,1.41%,1.87%,-27.14%,58%,"54,459,458"
5,JPM,+20.98%,+0.08%,0.70%,0.83%,-26.83%,58%,"10,778,886"
6,AMZN,+19.80%,+0.07%,1.01%,1.28%,-27.32%,56%,"40,845,408"
7,MSFT,+16.93%,+0.06%,0.77%,0.91%,-25.75%,56%,"27,230,177"
8,UNH,+13.17%,+0.05%,0.70%,0.85%,-24.78%,55%,"3,630,603"
9,TSLA,+2.38%,+0.02%,1.58%,2.19%,-41.48%,54%,"102,112,646"


## 5. Daily return correlation

Values near 1 move together, values near 0 behave independently, and negative values tend to move in opposite directions.

In [20]:
daily_returns = daily_all.pivot(index='day', columns='symbol', values='daily_return_pct')[symbols]
correlation = daily_returns.corr()
correlation_figure = go.Figure(go.Heatmap(z=correlation.values, x=symbols, y=symbols, zmin=-1, zmax=1, colorscale='RdBu', reversescale=True, text=correlation.round(2).values, texttemplate='%{text:.2f}', hovertemplate='%{y} vs %{x}: %{z:.3f}<extra></extra>', colorbar={'title': 'Correlation'}))
correlation_figure.update_layout(title=f'Daily return correlation: {year_label}', height=650, template='plotly_white', xaxis={'side': 'top'})
correlation_figure.show()

## 6. Average intraday pattern

For each stock and day, every bar is measured against that day's opening price, then averaged by time of day.

In [21]:
all_candles['day_open'] = all_candles.groupby(['symbol', 'day'])['open'].transform('first')
all_candles['session_return_pct'] = (all_candles['close'] / all_candles['day_open'] - 1) * 100
intraday = all_candles.groupby(['symbol', 'time'], as_index=False).agg(avg_return_pct=('session_return_pct', 'mean'))
intraday_figure = go.Figure()
for ticker in symbols:
    stock = intraday.loc[intraday['symbol'].eq(ticker)]
    intraday_figure.add_trace(go.Scatter(x=stock['time'], y=stock['avg_return_pct'], mode='lines', name=ticker, hovertemplate='%{x}<br>%{y:+.3f}%<extra>' + ticker + '</extra>'))
intraday_figure.add_hline(y=0, line_width=1, line_dash='dot', line_color='#64748b')
intraday_figure.update_layout(title=f'Average intraday return path: {year_label}', height=600, template='plotly_white', hovermode='x unified', xaxis_title='Chicago time', yaxis_title='Average return from daily open (%)', legend={'orientation': 'h', 'yanchor': 'bottom', 'y': 1.02, 'x': 0})
intraday_figure.show()

## 7. Selected-stock drill-down

In [22]:
figure = make_subplots(rows=2, cols=1, shared_xaxes=True, vertical_spacing=0.04, row_heights=[0.72, 0.28])
figure.add_trace(go.Candlestick(x=candles['timestamp'], open=candles['open'], high=candles['high'], low=candles['low'], close=candles['close'], name=symbol), row=1, col=1)
figure.add_trace(go.Bar(x=candles['timestamp'], y=candles['volume'], name='Volume', marker_color='#64748b'), row=2, col=1)
figure.update_layout(title=f'{symbol}: {month_label}', height=700, template='plotly_white', hovermode='x unified', xaxis_rangeslider_visible=False)
figure.update_yaxes(title_text='Price', row=1, col=1)
figure.update_yaxes(title_text='Volume', row=2, col=1)
figure.show()
daily_month = pd.to_datetime(daily_all['day']).dt.month.eq(month)
daily = daily_all.loc[daily_all['symbol'].eq(symbol) & daily_month, ['day', 'open', 'high', 'low', 'close', 'volume', 'trade_count', 'daily_return_pct', 'range_pct']].copy()
display(daily.style.format({'open': '{:.2f}', 'high': '{:.2f}', 'low': '{:.2f}', 'close': '{:.2f}', 'volume': '{:,.0f}', 'trade_count': '{:,.0f}', 'daily_return_pct': '{:+.2f}%', 'range_pct': '{:.2f}%'}))

,day,open,high,low,close,volume,trade_count,daily_return_pct,range_pct
0,2026-01-01,224.16,226.70,222.62,226.53,"55,011,804","23,400",+1.06%,1.83%
1,2026-01-02,226.53,227.77,225.49,226.89,"54,991,463","23,400",+0.16%,1.01%
2,2026-01-05,226.89,228.15,225.88,226.71,"54,998,248","23,400",-0.08%,1.01%
3,2026-01-06,226.70,227.02,222.27,222.72,"55,004,454","23,400",-1.76%,2.14%
4,2026-01-07,222.71,223.06,217.32,218.47,"55,005,451","23,400",-1.90%,2.64%
5,2026-01-08,218.46,218.59,216.41,217.26,"55,000,560","23,400",-0.55%,1.01%
6,2026-01-09,217.26,218.39,215.08,216.63,"54,995,531","23,400",-0.29%,1.54%
7,2026-01-12,216.64,217.60,215.00,216.91,"54,995,481","23,400",+0.12%,1.21%
8,2026-01-13,216.90,217.66,215.73,216.57,"54,994,635","23,400",-0.15%,0.89%
9,2026-01-14,216.56,217.18,213.10,213.82,"55,004,824","23,400",-1.27%,1.91%
